# SPLADE-v3 — Domain-Specific Keyphrase Ranking

Implementation inspired by:
> *SPLADE-v3: New baselines for SPLADE*  
> Formal et al., arXiv:2403.06789 (2024)

> *ERU-KG: Efficient Reference-aligned Unsupervised Keyphrase Generation*  
> Do et al., arXiv:2505.24219 (2025)

Model: [`naver/splade-v3`](https://huggingface.co/naver/splade-v3)

## Pipeline (inspired by ERU-KG)

```
Raw text
    ↓ chunking()           — overlapping windows of up to 512 model tokens
[SPLADE]                   — sparse vocabulary-term importance weights
    ↓                         w_j = max_i log(1 + ReLU(w_ij))
[spaCy noun phrases]       — candidate phrases from the original text
    ↓
[Re-ranking]               — average SPLADE weights of phrase tokens
    ↓                         f(c,x) = (1/|c|) Σ w_ci
Ranked keyphrases
```

## Key implementation decisions

| Step | Decision | Source / rationale |
|---|---|---|
| Term importance | SPLADE term weights | ERU-KG, §2.1.1 |
| Candidates | Noun phrases extracted with spaCy | ERU-KG, §2.2 (phraseness module; original uses POS-pattern parsing) |
| Phrase score | Mean SPLADE token weight | ERU-KG, Eq. 3 |
| Long texts | Overlapping chunks plus max pooling | BERT input-length limit |
| Absent keyphrases | Not implemented | Would require external pseudo-relevance feedback (ERU-KG, §2.1.3) |

This is an **ERU-KG-inspired pipeline**, not a complete reproduction of ERU-KG.

## 1. Dependencies

Uses **sentence-transformers** (`SparseEncoder`), **huggingface_hub**, **spaCy** (English model `en_core_web_sm`), **NumPy**, **pandas**, **matplotlib**, and **tqdm**. Initial installation and model downloads require internet access. A GPU is useful for larger corpora but not strictly required by the notebook design.

Version compatibility should be checked against your environment; exact versions from the original experiments have not been established.

In [ ]:
!pip install -q -U sentence-transformers huggingface_hub
!pip install -q spacy
!python -m spacy download en_core_web_sm -q

## 2. Hugging Face authentication

The original notebook used a Hugging Face token to access `naver/splade-v3`. Check the [model page](https://huggingface.co/naver/splade-v3) for its current access requirements and accept any required terms.

For local Jupyter: authenticate with `huggingface-cli login` / `hf auth login` or set an `HF_TOKEN` environment variable **outside the notebook**. In Google Colab, you can use a secret named `HF_TOKEN`. Never commit credentials to Git.

The next cell checks for a provided token and otherwise relies on existing local authentication. It never prompts automatically.

In [ ]:
# Optional authentication. Keep HF_TOKEN in your environment or Colab secrets.
import os
from huggingface_hub import login

token = os.getenv('HF_TOKEN')
if not token:
    try:
        from google.colab import userdata  # optional: Colab only
        token = userdata.get('HF_TOKEN')
    except (ImportError, AttributeError, KeyError):
        token = None
if token:
    login(token=token, add_to_git_credential=False)
    print('Hugging Face authentication configured.')
else:
    print('No HF_TOKEN supplied; using existing cached credentials or public access.')

## 3. Loading SPLADE-v3 and spaCy

In [ ]:
from sentence_transformers import SparseEncoder
import spacy

print('Loading naver/splade-v3 ...')
model = SparseEncoder('naver/splade-v3')
print('SPLADE-v3 loaded')

print('Loading spaCy model ...')
nlp = spacy.load('en_core_web_sm')
print('spaCy loaded')

## 4. `SPLADEKeyphraseExtractor` (ERU-KG-inspired)

**Phraseness module (spaCy):** extracts noun-phrase candidates from the original text. ERU-KG §2.2 instead describes a POS-pattern-based parser.

**Informativeness module (SPLADE):** assigns a weight to vocabulary tokens, inspired by ERU-KG §2.1.1.

**Aggregation (ERU-KG Eq. 3):** candidate phrases are scored using SPLADE weights of their constituent tokens, with the configurable length penalty in the implementation.

**Chunking:** documents longer than the model input limit are split into overlapping windows. Sparse weights are merged by **max pooling** across windows.

In [ ]:
import numpy as np
import re
from typing import List, Tuple, Dict
from collections import defaultdict


class SPLADEKeyphraseExtractor:
    """
    Keyphrase extractor ispirato a ERU-KG (Do et al., 2025).

    Pipeline:
    1. [Chunking]     — splits long texts into windows of MAX_TOKENS with overlap
    2. [SPLADE]       — computes term importances for each chunk
    3. [Max pooling]  — aggregates the weights of the chunks
    4. [spaCy NP]     — extracts noun phrase candidates from the original text
    5. [Re-ranking]   — sscore(NP) = average of the SPLADE scores of the component tokens
                        f(c,x) = 1/(|c|-gamma) * sum(w_ci)  [Eq. 3 ERU-KG]
    """

    MAX_TOKENS    = 512   # BERT limit
    CHUNK_OVERLAP = 64    # overlap between chunks in token
    LENGTH_PENALTY = 0.0  # gamma in Eq.3 (0=no penalty, negative=long phrases)

    STOPWORDS = {
        'the','a','an','is','are','was','were','be','been','being',
        'have','has','had','do','does','did','will','would','could',
        'should','may','might','shall','can','to','of','in','for',
        'on','with','at','by','from','as','it','its','this','that',
        'and','or','but','not','also','which','who','how','when',
        'where','what','there','their','they','we','our','i','you',
        'study','studies','patients','patient','results','conclusion',
        'methods','method','data','analysis','using','used','were',
        'among','between','compared','including','included','group'
    }

    def __init__(self, splade_model, spacy_nlp):
        self.model     = splade_model
        self.tokenizer = splade_model.tokenizer
        self.nlp       = spacy_nlp

    # CHUNKING: long texts handling

    def _split_into_chunks(self, text: str) -> List[str]:
        """
        Splits the text into chunks of MAX_TOKENS with an overlap of CHUNK_OVERLAP.
        Operates on BERT tokenizer tokens to respect exact limits.
        If the text is short, returns a single chunk.
        """
        tokens = self.tokenizer.encode(text, add_special_tokens=False)
        window = self.MAX_TOKENS - 2  # -2 for [CLS] and [SEP]
        step   = window - self.CHUNK_OVERLAP

        if len(tokens) <= window:
            return [text]

        chunks = []
        for start in range(0, len(tokens), step):
            chunk_ids  = tokens[start : start + window]
            chunk_text = self.tokenizer.decode(chunk_ids, skip_special_tokens=True)
            chunks.append(chunk_text)
            if start + window >= len(tokens):
                break
        return chunks


    # SPLADE: term importances with max pooling between chunks 

    def get_sparse_vector(self, text: str) -> np.ndarray:
        """
        Returns the sparse SPLADE vector (30522,) for the text
        For long texts: computes a vector for each chunk and applies max pooling
        """
        chunks = self._split_into_chunks(text)
        embeddings = self.model.encode_document(
            chunks, show_progress_bar=False
        )  # shape: (n_chunks, 30522)
        return embeddings.cpu().to_dense().numpy().max(axis=0)  # max pooling across chunks and conversion to NumPy

    def get_term_weights(self, text: str) -> Dict[str, float]:
        """Dictionary mapping token: SPLADE weight (only weights > 0)"""
        vec = self.get_sparse_vector(text)
        nonzero_ids = np.nonzero(vec)[0]
        return {
            self.tokenizer.convert_ids_to_tokens(int(idx)): float(vec[idx])
            for idx in nonzero_ids
        }

    # PHRASENESS MODULE: noun phrases extraction with spaCy        


    def _extract_noun_phrases(self, text: str) -> List[str]:
        """
        Extracts candidate noun phrases from the text using spaCy noun_chunks
        Corresponds to the phraseness module of ERU-KG (§2.2)
        """
        doc = self.nlp(text[:100_000])
        candidates = set()
        for chunk in doc.noun_chunks:
            phrase = chunk.text.lower().strip()
            # Removes leading articles/prepositions
            phrase = re.sub(r'^(the |a |an |of |in |for |on |with |at |by |from )', '', phrase).strip()
            words = phrase.split()
            if len(phrase) < 3:
                continue
            if all(w in self.STOPWORDS for w in words):
                continue
            if not any(c.isalpha() for c in phrase):
                continue
            candidates.add(phrase)
        return list(candidates)

    # INFORMATIVENESS: re-ranking with SPLADE score (Eq.3 ERU-KG)         

    def _score_phrase(self, phrase: str, term_weights: Dict[str, float]) -> float:
        """
        Eq. 3 of ERU-KG: f(c,x) = 1/(|c|-gamma) * sum(w_ci)
        where the component BERT tokens of phrase c are present
        """
        token_ids  = self.tokenizer.encode(phrase, add_special_tokens=False)
        token_strs = self.tokenizer.convert_ids_to_tokens(token_ids)
        scores = [term_weights.get(t, 0.0) for t in token_strs]
        if not scores:
            return 0.0
        length_factor = max(1, len(scores) - self.LENGTH_PENALTY)
        return sum(scores) / length_factor

    # CORE FUNCTION                                           

    def extract_keyphrases(
        self,
        text: str,
        top_k: int = 20,
        min_score: float = 0.1
    ) -> List[Tuple[str, float]]:
        """
        Extracts the top-k keyphrases (noun phrases) from a document:
        1. computes SPLADE term weights (with chunking for long texts)
        2. extracts noun phrase candidates with spaCy
        3. ranks the candidates by aggregated score (Eq. 3 ERU-KG)
        """
        term_weights = self.get_term_weights(text)
        candidates   = self._extract_noun_phrases(text)

        scored = [
            (phrase, self._score_phrase(phrase, term_weights))
            for phrase in candidates
        ]
        scored = [(p, s) for p, s in scored if s >= min_score]
        return sorted(scored, key=lambda x: x[1], reverse=True)[:top_k]


extractor = SPLADEKeyphraseExtractor(model, nlp)
print(f'   Chunking: {SPLADEKeyphraseExtractor.MAX_TOKENS} token windows, overlap {SPLADEKeyphraseExtractor.CHUNK_OVERLAP} token')

## 5. Corpus input: example or user file

The default `example` mode uses a short built-in corpus, so the notebook does not require the original research data. To use your own documents, set `CORPUS_SOURCE = 'file'` and update `CORPUS_PATH` in the next cell.

A local UTF-8 `.txt` file must contain **one document per nonempty line**, optionally formatted as `Title.. Abstract`. The research TXT corpora are intentionally not distributed with this repository.

Each document is processed separately; long inputs are handled by the extractors' chunking procedure.

In [ ]:
from pathlib import Path

# Option A: self-contained example, or Option B: UTF-8 .txt file (one document/line).
EXAMPLE_CORPUS = [
    'Machine learning studies algorithms that learn from training data to make predictions.',
    'Deep neural networks can classify images and analyze natural language.',
    'Keyphrase extraction identifies relevant concepts in scientific documents.',
    'Image segmentation assigns semantic labels to pixels in medical images.',
    'Researchers assess classification models using evaluation datasets and validation methods.'
]
CORPUS_SOURCE = 'example'  # 'example' or 'file'
CORPUS_PATH = Path('../data/ml_en_junior.txt')

if CORPUS_SOURCE == 'example':
    docs = EXAMPLE_CORPUS.copy()
elif CORPUS_SOURCE == 'file':
    if not CORPUS_PATH.is_file():
        raise FileNotFoundError(f'Corpus not found: {CORPUS_PATH.resolve()}')
    with CORPUS_PATH.open('r', encoding='utf-8') as f:
        docs = [line.strip() for line in f if line.strip()]
else:
    raise ValueError("CORPUS_SOURCE must be 'example' or 'file'")
if not docs:
    raise ValueError('The selected corpus is empty.')

# Token-length statistics for up to the first 100 documents.
tokenizer = extractor.tokenizer
token_counts = [len(tokenizer.encode(d, add_special_tokens=False)) for d in docs[:100]]
n_long = sum(tc > 510 for tc in token_counts)
print(f'Documents loaded: {len(docs)} (source: {CORPUS_SOURCE})')
print(f'Mean tokens/document (first {len(token_counts)}): {int(np.mean(token_counts))}')
print(f'Documents requiring chunking (first {len(token_counts)}): {n_long}')
print(f'\nFirst document: {docs[0][:300]}...')

## 6. Per-document keyphrase extraction and corpus aggregation

For each document:
1. SPLADE computes term weights (with chunking when necessary).
2. spaCy extracts noun-phrase candidates.
3. Candidates are re-ranked using the ERU-KG-inspired scoring function.

The rankings are then aggregated across the corpus.

In [ ]:
from tqdm.auto import tqdm

TOP_K_PER_DOC = 20   # noun phrases to extract per doc
MIN_SCORE     = 0.1  

corpus_scores  = defaultdict(list)
corpus_docfreq = defaultdict(int)
doc_results    = []

print(f'Analyzing {len(docs)} documents...')
for doc in tqdm(docs):
    kps = extractor.extract_keyphrases(doc, top_k=TOP_K_PER_DOC, min_score=MIN_SCORE)
    doc_results.append(kps)
    for phrase, score in kps:
        corpus_scores[phrase].append(score)
        corpus_docfreq[phrase] += 1

print(f'   Noun phrases extracted: {len(corpus_scores)}')

## 7. Corpus-level keyphrase ranking

In [ ]:
import pandas as pd

TOP_N = 30

rows = []
for phrase, scores in corpus_scores.items():
    rows.append({
        'keyphrase'   : phrase,
        'avg_score'   : np.mean(scores),
        'max_score'   : np.max(scores),
        'doc_freq'    : corpus_docfreq[phrase],
        'doc_freq_pct': corpus_docfreq[phrase] / len(docs) * 100
    })

df = pd.DataFrame(rows)
df['combined_score'] = df['avg_score'] * np.log1p(df['doc_freq'])

top_score    = df.sort_values('avg_score',     ascending=False).head(TOP_N)
top_freq     = df.sort_values('doc_freq',      ascending=False).head(TOP_N)
top_combined = df.sort_values('combined_score', ascending=False).head(TOP_N)

print('=== Top keyphrases per AVERAGE SCORE (ERU-KG informativeness) ===\n')
print(top_score[['keyphrase','avg_score','doc_freq','doc_freq_pct']]
      .to_string(index=False, float_format='%.3f'))

print('\n=== Top keyphrases per DOCUMENT FREQUENCY ===\n')
print(top_freq[['keyphrase','doc_freq','doc_freq_pct','avg_score']]
      .to_string(index=False, float_format='%.3f'))

print('\n=== Top keyphrases per COMBINED SCORE (score x log_freq) ===\n')
print(top_combined[['keyphrase','combined_score','avg_score','doc_freq','doc_freq_pct']]
      .to_string(index=False, float_format='%.3f'))

## 8. Visualizations

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(22, 8))
titles = ['Avg SPLADE Score', 'Document Frequency (%)', 'Combined Score']
cols   = ['avg_score', 'doc_freq_pct', 'combined_score']
tops   = [top_score, top_freq, top_combined]
colors = ['#3498db', '#2ecc71', '#e74c3c']

for ax, title, col, top, color in zip(axes, titles, cols, tops, colors):
    top_n = top.head(20)
    ax.barh(top_n['keyphrase'][::-1], top_n[col][::-1], color=color, alpha=0.85)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel(col, fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='x', alpha=0.3)
    ax.tick_params(axis='y', labelsize=9)

plt.suptitle('SPLADE-v3 + spaCy — Keyphrase Extraction (ERU-KG style)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('splade_corpus_keyphrases.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Individual-document analysis

In [ ]:
N_DOCS_TO_SHOW = 5

for i in range(min(N_DOCS_TO_SHOW, len(docs))):
    n_tokens = len(extractor.tokenizer.encode(docs[i], add_special_tokens=False))
    n_chunks = len(extractor._split_into_chunks(docs[i]))
    chunk_info = f'{n_tokens} token' + (f' -> {n_chunks} chunk' if n_chunks > 1 else '')
    print(f'\n{"="*65}')
    print(f'📄 Doc {i+1}  [{chunk_info}]')
    print(f'   {docs[i][:120].strip()}...')
    print(f'\n🔑 Top keyphrases (noun phrases re-ranked with SPLADE):')
    print(f'   {"Keyphrase":<38} {"Score"}')
    print(f'   {"-"*48}')
    for kp, score in doc_results[i][:10]:
        print(f'   {kp:<38} {score:.4f}')

## 10. Notes and limitations

| Aspect | Notes |
|---|---|
| **Long documents** | 512-token windows with overlap of 64, followed by max pooling across chunks |
| **Phraseness** | spaCy extracts noun phrases from the original text |
| **Informativeness** | SPLADE assigns token weights, aggregated to candidate-phrase scores (ERU-KG-inspired Eq. 3) |
| **Absent keyphrases** | Not implemented; would require pseudo-relevance feedback with an external corpus (ERU-KG §2.1.3) |
| **`LENGTH_PENALTY`** | Length correction in candidate scoring; negative values favor longer phrases |
| **Combined score** | `avg_score × log(1 + doc_freq)` combines SPLADE relevance and corpus coverage |

**References:** Formal et al., *SPLADE-v3: New baselines for SPLADE* (2024); Do et al., *ERU-KG: Efficient Reference-aligned Unsupervised Keyphrase Generation* (2025).

## 11. Strengths and limitations

**Strengths**
- Produces sparse vocabulary-weight vectors, useful for interpretable lexical weighting and efficient storage.
- The ERU-KG-inspired design separates **phraseness** (spaCy noun phrases) from **informativeness** (SPLADE token weights).
- Max pooling across overlapping chunks supports long documents.
- Per-document term weights can be aggregated across a corpus.

**Limitations**
- **Domain mismatch:** SPLADE-v3 was trained on general-domain retrieval data; specialized technical terminology may be weighted poorly. The original experiments suggested this limitation, but no ground-truth accuracy benchmark is provided here.
- **Domain adaptation cost:** fine-tuning requires suitable retrieval training data not supplied with these corpora.
- **Max-pooling saturation:** a single high weight in one chunk can dominate the aggregate weight.
- Model downloading and any access restrictions may complicate use on computing clusters.